# 4.9 MCP 客户端实战

对应笔记 **4.7.2 MCP客户端**。

本 notebook 带你亲手搭建一个 **MCP 客户端**,并按笔记的演进路径走一遍:

1. **模拟客户端** —— 笔记原版的骨架(会话 + 资源管理 + 聊天循环);
2. **编写 MCP Server** —— 用 MCP SDK 暴露真实工具(天气查询 + 加法);
3. **连接真实服务器** —— 客户端通过 stdio 连接 server,动态发现工具并调用;
4. **接入在线模型(Qwen)** —— 让大模型通过 MCP 客户端调用远程工具,复用 4.8 的 Function calling 机制。

> 用真实 SDK(MCP 2.0)+ 阿里云 DashScope 的 Qwen 演示。
> 环境:在 `llm-finetune` conda 环境里已安装 `mcp`。

## 0. 环境准备

```bash
# 已在你机器的 llm-finetune 环境安装:
d:/CondaData/envs/llm-finetune/python.exe -m pip install mcp
```

> ⚠️ 注意:笔记里写的是 MCP SDK 1.x 的 `mcp.server.fastmcp` 用法(如 `@mcp.tool()` 装饰器)。你装到的是 **MCP 2.0**,该版本把 `FastMCP` 从包中移除了,改用底层 `Server` + 回调。本 notebook 按 **2.0 的实际 API** 编写,原理与笔记一致。

In [1]:
import asyncio
import json
import os
import sys
import tempfile
import re
import subprocess
import threading
from contextlib import AsyncExitStack

from openai import OpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


# ==================== 关键:MCP stdio 稳定运行器 ====================
# 坑:VS Code 的 ipykernel 内核启动时就创建了 Windows Proactor 事件循环,
# 而 MCP 2.0 的 stdio_client 在 Proactor 下启动子进程会卡死(握手挂起)。
# set_event_loop_policy 无法改变已存在的 loop —— 所以用一个独立线程 +
# 独立 Selector 事件循环来跑所有 MCP 异步操作,彻底绕开内核的 loop。
def mcp_run(coro_factory):
    """在独立线程的 Selector 事件循环里运行 MCP async 任务。

    参数 coro_factory 是无参函数,返回一个协程对象。
    返回 coro_factory 的结果;若抛异常则向上抛出。
    """
    result = {}

    def worker():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
        inner = asyncio.new_event_loop()
        asyncio.set_event_loop(inner)
        try:
            result["val"] = inner.run_until_complete(coro_factory())
        except Exception as e:
            result["err"] = e
        finally:
            inner.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "err" in result:
        raise result["err"]
    return result["val"]


print("✅ 导入成功")
print("Python:", sys.executable)

✅ 导入成功
Python: d:\CondaData\envs\llm-finetune\python.exe


## 1. 模拟客户端(笔记原版)

笔记第一步是搭建一个「先不连服务器」的客户端骨架,理解三个核心概念:

- **`ClientSession`**(会话):可获取外部工具列表、保存会话状态;
- **`AsyncExitStack`**(资源管理):确保客户端关闭时释放资源;
- **`chat_loop`**(聊天循环):命令行交互界面。

下面的类与笔记完全一致。

In [2]:
class MCPClient:
    def __init__(self):
        """初始化 MCP 客户端"""
        # 核心概念:会话,可以获取外部工具列表,保存当前会话状态等功能,暂时不链接MCP服务器
        self.session = None
        # 异步通信资源管理器
        self.exit_stack = AsyncExitStack()

    async def connect_to_mock_server(self):
        """模拟 MCP 服务器的连接(暂不连接真实服务器)"""
        print("✅ MCP 客户端已初始化,但未连接到服务器")

    async def chat_loop(self):
        """运行交互式聊天循环"""
        print("\nMCP 客户端已启动!输入 'quit' 退出")
        while True:
            try:
                query = input("\nQuery: ").strip()
                if query.lower() == 'quit':
                    break
                print(f"\n🤖 [Mock Response] 你说的是:{query}")
            except Exception as e:
                print(f"\n⚠️ 发生错误: {str(e)}")

    async def cleanup(self):
        """清理资源"""
        await self.exit_stack.aclose()  # 关闭资源管理器

`chat_loop` 里用了 `input()` 从终端读输入,notebook 里没法交互。下面用**一组模拟输入**跑一遍同样的逻辑,让你看到「判断 quit → 回显 → 清理」的流程。

In [3]:
async def run_mock_demo():
    client = MCPClient()
    await client.connect_to_mock_server()
    print("\nMCP 客户端已启动!输入 'quit' 退出")
    # 笔记里用 input() 读终端;notebook 里用模拟输入代替
    for query in ["你好", "北京天气怎么样?", "quit"]:
        print(f"\nQuery: {query}")
        if query.lower() == "quit":
            break
        print(f"\n🤖 [Mock Response] 你说的是:{query}")
    await client.cleanup()

# notebook 内核已有事件循环,直接 await 而不是 asyncio.run()
await run_mock_demo()

✅ MCP 客户端已初始化,但未连接到服务器

MCP 客户端已启动!输入 'quit' 退出

Query: 你好

🤖 [Mock Response] 你说的是:你好

Query: 北京天气怎么样?

🤖 [Mock Response] 你说的是:北京天气怎么样?

Query: quit


## 2. 编写 MCP Server

笔记 4.7.1 给出了一个「简单服务器示例」(`@mcp.tool()` 装饰器)。MCP 2.0 改成**底层 `Server` + 回调**方式,职责完全对应:

| 笔记 1.x 示例 | MCP 2.0 写法 | 作用 |
|--------------|-------------|------|
| `mcp = FastMCP("Demo")` | `Server("Demo", on_list_tools=..., on_call_tool=...)` | 创建服务器 |
| `@mcp.tool()` 装饰函数 | `on_list_tools` + `on_call_tool` 回调 | 注册工具、实现执行 |

我们把服务器代码写成一个独立脚本,之后让客户端用 **stdio 子进程**去连接它(这正符合"服务器是独立运行环境"的定位)。服务器暴露两个工具:**天气查询** + **加法**(呼应 4.7.1 示例里的 `add`)。

In [4]:
# 服务器源码:MCP 2.0 底层 Server 写法
SERVER_CODE = r'''
import asyncio, json
from mcp.server.lowlevel.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent, ListToolsResult, CallToolResult

# 服务器本地数据源
WEATHER_DB = {
    "北京":   {"weather": "晴", "temperature": 32},
    "杭州":   {"weather": "小雨", "temperature": 27},
    "余杭区": {"weather": "阴", "temperature": 26},
    "上海":   {"weather": "多云", "temperature": 30},
}

# 1) 工具列表:客户端连接后"动态发现"
async def on_list_tools(ctx, params):
    return ListToolsResult(tools=[
        Tool(
            name="get_current_weather",
            description="当你想查询指定城市的天气时非常有用。",
            inputSchema={
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "城市或县区,比如北京市、杭州市、余杭区等。"}
                },
                "required": ["location"],
            },
        ),
        Tool(
            name="add",
            description="计算两个整数之和。",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "integer", "description": "第一个加数"},
                    "b": {"type": "integer", "description": "第二个加数"},
                },
                "required": ["a", "b"],
            },
        ),
    ])

# 2) 工具执行:客户端调用后,这里真正运行
async def on_call_tool(ctx, params):
    name = params.name
    args = params.arguments or {}
    if name == "get_current_weather":
        location = args.get("location", "")
        key = location.rstrip("市")  # 规范化:模型常输出"北京市"
        data = WEATHER_DB.get(key, {"weather": "未知", "temperature": None})
        text = json.dumps({"city": location, **data}, ensure_ascii=False)
        return CallToolResult(content=[TextContent(type="text", text=text)])
    if name == "add":
        total = int(args.get("a", 0)) + int(args.get("b", 0))
        return CallToolResult(content=[TextContent(type="text", text=str(total))])
    return CallToolResult(content=[TextContent(type="text", text=f"未知工具: {name}")])

server = Server("Demo", on_list_tools=on_list_tools, on_call_tool=on_call_tool)

async def main():
    # stdio 服务器:通过标准输入输出与客户端通信
    async with stdio_server() as (read, write):
        await server.run(read, write, server.create_initialization_options())

asyncio.run(main())
'''

# 写入临时脚本文件,供 stdio 子进程启动
SERVER_PATH = os.path.join(tempfile.gettempdir(), "mcp_server_demo.py")
with open(SERVER_PATH, "w", encoding="utf-8") as f:
    f.write(SERVER_CODE)
print("✅ 服务器脚本已写入:", SERVER_PATH)

✅ 服务器脚本已写入: C:\Users\ADMINI~1\AppData\Local\Temp\mcp_server_demo.py


## 3. 客户端连接真实服务器(stdio)

现在让客户端**真的连上**服务器。流程:

1. `StdioServerParameters` 描述怎么启动服务器子进程(用哪个 python 跑哪个脚本);
2. `stdio_client()` 建立 stdio 双向管道;
3. `ClientSession.initialize()` 握手,`list_tools()` **动态发现**服务器暴露的工具;
4. `call_tool(name, args)` 调用具体工具。

这就是笔记说的:MCP 把"外部函数运行环境"封装成 server,**客户端不用预先知道有哪些函数**。

In [5]:
def connect_and_explore():
    """连接 MCP 服务器并浏览工具(用 mcp_run 在独立 Selector loop 里跑)"""
    async def session_coro():
        params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
        # 关键:notebook 内核的 sys.stderr 不支持 fileno,必须显式传 errlog=PIPE
        async with stdio_client(params, errlog=subprocess.PIPE) as (read, write):
            async with ClientSession(read, write) as session:
                # 1) 握手
                await session.initialize()
                print("✅ 已连接到 MCP 服务器")

                # 2) 动态发现工具(客户端此前并不知道有这些工具!)
                tools = await session.list_tools()
                print("\n📋 服务器暴露的工具:")
                for t in tools.tools:
                    print(f"  - {t.name}: {t.description}")
                    # MCP 2.0 的 Tool 属性是 snake_case:input_schema
                    print(f"      参数: {json.dumps(t.input_schema, ensure_ascii=False)}")

                # 3) 调用天气工具
                print("\n🌤️ 调用 get_current_weather(location='杭州'):")
                res = await session.call_tool("get_current_weather", {"location": "杭州"})
                print("   返回:", res.content[0].text)

                # 4) 调用加法工具
                print("\n➕ 调用 add(a=3, b=5):")
                res = await session.call_tool("add", {"a": 3, "b": 5})
                print("   返回:", res.content[0].text)

    return mcp_run(session_coro)

connect_and_explore()

✅ 已连接到 MCP 服务器

📋 服务器暴露的工具:
  - get_current_weather: 当你想查询指定城市的天气时非常有用。
      参数: {"properties": {"location": {"type": "string", "description": "城市或县区,比如北京市、杭州市、余杭区等。"}}, "required": ["location"], "type": "object"}
  - add: 计算两个整数之和。
      参数: {"properties": {"a": {"type": "integer", "description": "第一个加数"}, "b": {"type": "integer", "description": "第二个加数"}}, "required": ["a", "b"], "type": "object"}

🌤️ 调用 get_current_weather(location='杭州'):
   返回: {"city": "杭州", "weather": "小雨", "temperature": 27}

➕ 调用 add(a=3, b=5):
   返回: 8


## 4. 接入在线模型(Qwen)

上一节是客户端手动调工具。现在把 **4.8 的 Function calling 循环**接到 MCP 上,让大模型自己决定调什么工具:

```
用户提问 → Qwen 判断 → 输出 <tool_call> → 客户端经 MCP 会话调用 server 工具
         → 结果封装 <tool_response> 回传 → Qwen 生成最终答案
```

**与 4.8 的本质区别**:4.8 的 `FUNCTIONS` 注册表是**硬编码**的;这里工具列表是客户端**从 MCP server 动态发现**的 —— 换个服务器,客户端代码一行不用改。这正是 MCP 的价值。

> 用你的 `DASHSCOPE_API_KEY`(Qwen)走 DashScope 兼容接口,与 4.8 一致。

In [6]:
# ==================== 配置 ====================
client_llm = OpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY") or "sk-xxxxxxxx",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
MODEL = "qwen-plus"


# ==================== 工具调用解析(复用 4.8 的机制) ====================
def parse_tool_calls(text: str) -> list[dict]:
    """用正则解析出所有 <tool_call>...</tool_call> 里的 JSON 对象。"""
    pattern = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)
    calls = []
    for match in pattern.finditer(text):
        try:
            calls.append(json.loads(match.group(1)))
        except json.JSONDecodeError as e:
            print("⚠️ JSON 解析失败:", e)
    return calls


def chat_once(messages: list) -> str:
    """一次模型调用(非流式)。"""
    resp = client_llm.chat.completions.create(model=MODEL, messages=messages)
    return resp.choices[0].message.content


# ==================== MCP 客户端 + 在线模型 ====================
# mcp_run 已在 §0 imports 里定义:在独立线程的 Selector loop 里跑 MCP 会话。
class MCPClient:
    def __init__(self, server_script: str):
        """初始化 MCP 客户端:记住服务器启动方式"""
        self.server_params = StdioServerParameters(command=sys.executable, args=[server_script])

    def process_query(self, query: str, verbose: bool = True) -> str:
        """完整一轮:用户问题 → 模型判工具 → MCP 执行 → 模型作答"""
        return mcp_run(lambda: self._session_coro(query, verbose))

    async def _session_coro(self, query: str, verbose: bool):
        """一个完整的 MCP 会话(在 mcp_run 的独立 Selector loop 里执行)"""
        # 关键:notebook 内核的 sys.stderr 不支持 fileno,必须显式传 errlog=PIPE
        async with stdio_client(self.server_params, errlog=subprocess.PIPE) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()

                # 把动态发现的工具拼成 system 提示词(相当于把 JSON Schema 喂给模型)
                lines = [
                    "你是一个会调用外部工具的智能体。",
                    "当你需要查询实时数据或进行计算时,必须严格按照以下格式输出工具调用:",
                    "<tool_call>",
                    '{"name": "函数名", "arguments": {"参数1": "值"}}',
                    "</tool_call>",
                    "可用工具如下:",
                ]
                for t in tools.tools:
                    lines.append(json.dumps(
                        {"name": t.name, "description": t.description, "parameters": t.input_schema},
                        ensure_ascii=False, indent=2,
                    ))
                system_prompt = "\n".join(lines)

                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": query},
                ]

                # 1) 模型判定是否调用工具(同步 openai 调用放进 executor,避免阻塞 event loop)
                answer = await asyncio.get_event_loop().run_in_executor(
                    None, lambda: chat_once(messages))
                messages.append({"role": "assistant", "content": answer})

                # 2) 解析工具调用
                calls = parse_tool_calls(answer)
                if not calls:
                    return answer  # 模型认为不需要工具

                # 3) 通过 MCP 会话执行(而不是硬编码的 FUNCTIONS!)
                results = []
                for call in calls:
                    res = await session.call_tool(call["name"], call["arguments"])
                    result = res.content[0].text
                    results.append({"tool": call["name"], "result": result})
                    if verbose:
                        print(f"🤖 模型要调用: {call['name']}{call['arguments']}")
                        print(f"🛠️  MCP 服务器返回: {result}")

                # 4) 封装 tool_response,以 user 角色回传
                messages.append({
                    "role": "user",
                    "content": "<tool_response>\n"
                    + "\n".join(json.dumps(r, ensure_ascii=False) for r in results)
                    + "\n</tool_response>",
                })

                # 5) 模型基于工具结果生成最终答案
                return await asyncio.get_event_loop().run_in_executor(
                    None, lambda: chat_once(messages))

连上服务器,用 Qwen 问几个不同的问题,观察模型什么时候决定调工具、什么时候直接回答:

In [7]:
client = MCPClient(SERVER_PATH)

queries = [
    "北京今天天气怎么样?",
    "帮我算一下 3 + 5 等于多少?",
    "你好呀,介绍一下你自己",
]
for q in queries:
    print(f"\n{'='*50}\n👤 你: {q}")
    answer = client.process_query(q)  # 同步调用,mcp_run 在内部独立 loop 里跑
    print(f"🤖 Qwen: {answer}\n{'='*50}")


👤 你: 北京今天天气怎么样?
🤖 模型要调用: get_current_weather{'location': '北京市'}
🛠️  MCP 服务器返回: {"city": "北京市", "weather": "晴", "temperature": 32}
🤖 Qwen: 北京今天天气晴，气温为32℃。

👤 你: 帮我算一下 3 + 5 等于多少?
🤖 模型要调用: add{'a': 3, 'b': 5}
🛠️  MCP 服务器返回: 8
🤖 Qwen: 3 + 5 等于 8。

👤 你: 你好呀,介绍一下你自己
🤖 Qwen: 你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊


## 📊 总结

### MCP 客户端全流程(对照笔记 4.7.2)

| 环节 | 笔记内容 | 本 notebook 实现 |
|------|---------|-----------------|
| **会话管理** | `ClientSession` 获取工具列表、保存会话状态 | `session.initialize()` + `list_tools()` |
| **资源管理** | `AsyncExitStack` 确保释放资源 | `async with` 自动退出 + `cleanup()` |
| **动态发现工具** | 客户端不用预先知道有哪些函数 | `list_tools()` 从 server 拉取工具 Schema |
| **调用工具** | 按 `name` 执行 | `session.call_tool(name, arguments)` |
| **接入在线模型** | 大模型通过 client 调用远程工具 | Qwen + `<tool_call>` 解析 + `<tool_response>` 回传 |

### 关键收获:MCP vs 裸 Function calling

- **4.8 里** `FUNCTIONS = {...}` 是硬编码的注册表,工具多了要手动维护;
- **MCP 里** 工具由 server 动态暴露,client 用 `list_tools()` 发现 —— 换服务器不用改客户端代码;
- MCP 把「工具注册表」标准化成了**协议**:同一个 client 可以连任何符合 MCP 规范的 server,这就是笔记说的"避免外部函数重复编写"。

### ⚠️ MCP 2.0 与笔记 1.x 的差异

笔记示例用 `mcp.server.fastmcp.FastMCP`(`@mcp.tool()` 装饰器)。MCP **2.0 移除了 FastMCP**,改用 `Server("Demo", on_list_tools=..., on_call_tool=...)` 底层写法。两者理念一致,2.0 给了更大的控制力。若你以后想看 1.x 的 FastMCP 用法,可 `pip install "mcp<2"` 单独体验。

> 提示:想连「网上现成的 MCP 服务器」,把 `StdioServerParameters` 换成对应的启动命令(如 `npx` 或远程 URL)即可。